# Test the pre-trained DLM on Colab

Loads the model you trained (from Google Drive), generates Stories, and measures it. It does **no training**. Run the cells top to bottom.

- Works with a **finished** run (`final/`) and with an **unfinished / stopped** run (it makes `final/` from `resume.pt`; the checkpoint is then an intermediate one).
- A GPU is not required (the model is small), but **Runtime -> Change runtime type -> GPU** makes it faster.
- Cell 1 has `QUIT_WHEN_DONE = True`: the last cell ends the session when the evaluation report is saved on Drive, so no credits are spent while you read. Set it to `False` if you want to keep experimenting.
- Uploading to Hugging Face is **off** (`PUSH_TO_HUB = False`) and private when on.

In [ ]:
# 1. Settings + Google Drive
RUN_NAME = "pretrain_base"  # which training run to test: the folder under checkpoints/ ("pretrain_cosine" for the cosine run)
SCHEDULE = "linear"       # the noise schedule that run was trained with ("linear" or "cosine"); generation uses the same one
RUN_EVAL = True           # cell 9: measure likelihood + sample statistics on a validation subset (needs the token cache on Drive; ~5-10 min)
QUIT_WHEN_DONE = True     # the LAST cell ends the Colab session once the model and the evaluation report are on Drive
GRACE_SECONDS = 60        # countdown before the session ends (stop the last cell during it to cancel)
PUSH_TO_HUB = False       # True = upload to YOUR Hugging Face account in cell 10 (needs a Colab secret named HF_TOKEN with write access)
HF_REPO_ID = ""           # e.g. "your-username/dlm-tinystories-36m"
HF_PRIVATE = True         # keep True until you have looked at the repo on huggingface.co
HF_DRY_RUN = True         # True = build and check the upload folder only, upload nothing. Set False for the real upload
HF_LICENSE = "apache-2.0" # written into the model card: choose yours before making the repo public

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/diffusion_lm_outputs"
RUN = f"{OUT}/checkpoints/{RUN_NAME}"
CKPT = f"{RUN}/final"

In [ ]:
# 2. Get the latest code from GitHub (public repo; tokenizer/ is committed there). Re-running this cell is safe.
REPO = "https://github.com/bitlabsdevteam/Diffusion-Language-Model.git"
import os
if os.path.isdir("/content/dlm_repo"):
    !git -C /content/dlm_repo pull --ff-only
else:
    !git clone -q $REPO /content/dlm_repo
%cd /content/dlm_repo
!git log --oneline -1

In [ ]:
# 3. Dependencies: install ONLY what is missing (never numpy / torch: reinstalling them can break Colab's live kernel)
import importlib.util, subprocess, sys
missing = [p for p in ("tokenizers", "safetensors", "huggingface_hub", "pyarrow", "datasets") if importlib.util.find_spec(p) is None]
print("missing:", missing or "none")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def sh(*args):
    """Run `python run.py <args>` and print its output."""
    p = subprocess.run([sys.executable, "run.py", *args], capture_output=True, text=True)
    print(p.stdout + p.stderr)
    return p.returncode

In [ ]:
# 4. Find the model to test. A finished run has final/. A stopped run only has resume.pt: export it to final/ (about 10 seconds).
import json, os
if os.path.isfile(f"{CKPT}/model.pt"):
    print("Using the model in", CKPT)
elif os.path.isfile(f"{RUN}/resume.pt"):
    print("No final/ yet (the run was stopped before the end): exporting resume.pt")
    if sh("export", "--resume", f"{RUN}/resume.pt", "--out", CKPT) != 0:
        raise SystemExit("export failed (see above)")
else:
    raise SystemExit(f"Nothing to test: neither {CKPT}/model.pt nor {RUN}/resume.pt exists. Is Drive mounted and the run folder right?")
for name in ("export.json", "run_info.json"):
    p = f"{CKPT}/{name}"
    if os.path.isfile(p):
        d = json.load(open(p))
        d.pop("recipe", None)
        print(name, "->", d)

In [ ]:
# 5. The training curves (cross-entropy loss, learning rate, epoch, grad norm). The gold star marks the best validation point.
from IPython.display import Image, display
if os.path.isfile(f"{RUN}/training_curves.png"):
    display(Image(f"{RUN}/training_curves.png"))
else:
    print("No training_curves.png in the run folder.")

In [ ]:
# 6. Three Stories from scratch (no prompt). Each line below the Story is its length and whether it finished.
sh("generate", "--ckpt", CKPT, "--n", "3", "--seed", "0", "--schedule", SCHEDULE)

In [ ]:
# 7. Continue YOUR opening words (edit the form and re-run as often as you like). --trace shows the unmasking as text rows
# (every row = one step; blocks of 32 are filled left to right, most confident first). Avoid double quotes in the prompt.
PROMPT = "Once upon a time, there was a little"  #@param {type:"string"}
HOW_MANY = 2  #@param {type:"integer"}
SEED = 0  #@param {type:"integer"}
TEMPERATURE = 0.8  #@param {type:"number"}
import glob, os
from IPython.display import Image, display
for f in glob.glob("/content/unmask*.gif"):
    os.remove(f)
sh("generate", "--ckpt", CKPT, "--prompt", PROMPT, "--n", str(HOW_MANY), "--seed", str(SEED),
   "--temperature", str(TEMPERATURE), "--schedule", SCHEDULE, "--trace", "--gif", "/content/unmask.gif")
gifs = sorted(glob.glob("/content/unmask*.gif"))
if gifs:
    display(Image(gifs[0]))   # the unmasking animation of the first sample

In [ ]:
# 8. Does decoding speed or randomness change quality? Same opening words: greedy (temperature 0, deterministic), the default,
# and a fast setting (60 steps for ~480 tokens = about 8 tokens per step). Compare the text and the printed time.
import time
for label, extra in (("greedy", ["--temperature", "0"]), ("default", []), ("fast, 60 steps", ["--steps", "60"])):
    print(f"\n===== {label} =====")
    t0 = time.time()
    sh("generate", "--ckpt", CKPT, "--prompt", PROMPT, "--n", "1", "--seed", "1", "--schedule", SCHEDULE, *extra)
    print(f"[{label}: {time.time() - t0:.1f} s including start-up]")

In [ ]:
# 9. Measure the model (optional, RUN_EVAL): validation likelihood next to the 5.92 token-frequency baseline, and statistics of
# 50 generated Stories vs real ones (finished rate, length, distinct-n, repetition, collapse). A QUICK version on a subset of
# the validation blocks (writes <run>/eval/report.md + samples.md on Drive). Needs the token cache from the data cells of run_colab.ipynb.
if not RUN_EVAL:
    print("Skipped (RUN_EVAL = False).")
elif not os.path.isfile(f"{OUT}/data/tokens/tokens_manifest.json"):
    print(f"Skipped: no token cache at {OUT}/data/tokens (run cells 4-6 of run_colab.ipynb once, or set RUN_EVAL = False).")
else:
    !mkdir -p /content/tokens && cp -r $OUT/data/tokens/. /content/tokens/
    !python run.py eval --ckpt $CKPT --cache /content/tokens --n-stories 50 --n-prompts 10 --draws 2 --max-val-blocks 1500 --no-gpt2
    print(open(f"{RUN}/eval/report.md").read())
    print(open(f"{RUN}/eval/samples.md").read()[:5000])

In [ ]:
# 10. (optional) Publish to the Hugging Face Hub, private by default. SKIPPED unless PUSH_TO_HUB = True. Needs a Colab secret named
# HF_TOKEN (key icon in the left sidebar; token with WRITE access; switch on "Notebook access"). With HF_DRY_RUN = True it only builds
# and checks the upload folder (model.safetensors, config.json, tokenizer, model card). A model made from resume.pt gets a card that
# says it is an intermediate checkpoint. An error here is printed and never stops the next cell.
if not PUSH_TO_HUB:
    print("Skipping the Hugging Face step (PUSH_TO_HUB = False in cell 1).")
elif not HF_REPO_ID:
    print("NOT uploading: set HF_REPO_ID in cell 1, for example 'your-username/dlm-tinystories-36m'.")
else:
    try:
        from dlm.hub import publish
        publish(RUN, repo_id=HF_REPO_ID, private=HF_PRIVATE, license=HF_LICENSE, out_dir="/content/hf_upload", dry_run=HF_DRY_RUN)
    except Exception as e:
        print(f"NOT uploaded ({type(e).__name__}: {e}). The model is safe on Drive.")

In [ ]:
# 11. END THE SESSION (only when the model, and the evaluation report if RUN_EVAL, are saved on Drive).
try:
    from dlm.colab import finish_session
    finish_session(OUT, enabled=QUIT_WHEN_DONE, grace_seconds=GRACE_SECONDS, require_eval=RUN_EVAL)
except Exception as e:  # never end the session by accident
    print(f"NOT ending the session ({type(e).__name__}: {e}). Stop it yourself: Runtime -> Disconnect and delete runtime.")